# Experiment 1 — Dataset audit & boundedness

**Paper reference.** Appendix B.1 (v7 spec), Tables 3 and 4.

**Goal.** Verify data integrity, compute pooled turn counts per
(policy, vehicle) after the v7 preprocessing, and document truncation
rates for both bounds separately. Also provide a reader-facing view of
the raw negotiation dynamics.

## 1. Setup and data load

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
runs_per_policy = raw.groupby('policy')['run_id'].nunique().to_dict()
print(f'Loaded {len(raw):,} raw turn records.')
print(f'Runs per policy: {runs_per_policy}')

## 2. Vehicle key

Five vehicles, labelled `Vehicle 1`..`Vehicle 5` in ascending order of
the margin ratio $v/M$ (low ratio = lots of room to negotiate).

In [ ]:
car_info = (raw.groupby('car_name')
                 .agg(v=('v', 'first'), M=('M', 'first'))
                 .assign(ratio=lambda x: x['v'] / x['M'])
                 .sort_values('ratio'))
car_info['vehicle'] = [f'Vehicle {i+1}' for i in range(len(car_info))]
car_info[['vehicle', 'v', 'M', 'ratio']]

## 3. Apply v7 preprocessing

- Truncate $S_t$ to $[v, M]$
- Mark first agreement: $S_t \le 1.01 B_t$
- Drop all turns *after* the first agreement in each run

In [ ]:
all_marked, turns = nu.preprocess_turns(raw)
n_dropped = int(all_marked['after_agreement'].sum())
n_pf      = int(turns['parse_fail'].sum())
n_clean   = int((~turns['parse_fail']).sum())
print(f'Post-agreement dropped: {n_dropped:,}')
print(f'Parse failures (kept for lag alignment): {n_pf:,}')
print(f'Clean analysis turns: {n_clean:,}')

## 4. Table 3 — pooled negotiation turns

Each cell counts parse-success turns for that (policy, vehicle)
combination, after truncation and agreement cutoff.

In [ ]:
POLICY_ORDER = list(nu.POLICIES)
VEHICLE_COLS = [f'Vehicle {i}' for i in range(1, 6)]

table3 = (turns[~turns['parse_fail']]
            .groupby(['policy', 'vehicle']).size()
            .unstack('vehicle')
            .reindex(index=POLICY_ORDER, columns=VEHICLE_COLS)
            .fillna(0).astype(int))
table3['Total'] = table3.sum(axis=1)
table3.loc['Total'] = table3.sum(axis=0)
table3

## 5. Table 4 — truncation proportions

Percentages are computed relative to the turn counts in Table 3.
`lower_truncated` counts turns where the raw seller price was below $v$
(irrational underpricing); `upper_truncated` counts turns with raw price
above $M$ (irrational overpricing). Both types are treated as instruction
non-adherence and clipped to the feasible interval before analysis.

In [ ]:
clean = turns[~turns['parse_fail']]
lo = (clean.groupby(['policy', 'vehicle'])['lower_truncated'].mean() * 100
           ).unstack('vehicle').reindex(index=POLICY_ORDER, columns=VEHICLE_COLS).round(1)
up = (clean.groupby(['policy', 'vehicle'])['upper_truncated'].mean() * 100
           ).unstack('vehicle').reindex(index=POLICY_ORDER, columns=VEHICLE_COLS).round(1)

table4 = pd.concat({'S_t < v (%)': lo, 'S_t > M (%)': up}, axis=1)
table4

## 6. Per-vehicle negotiation trajectories

A 2x3 panel: the first five subplots show, for each vehicle, the mean
seller ask per round under each of the six policies with a $\pm 1$ SD
band. The sixth subplot shows the distribution of normalized asks by
policy (a single pooled violin, since raw-dollar ranges differ by car).

In [ ]:
df_plot = turns[~turns['parse_fail']].copy()

cars      = car_info.index.tolist()   # already sorted by v/M ratio
policies  = POLICY_ORDER
cmap      = plt.get_cmap('tab10')
colors    = {p: cmap(i) for i, p in enumerate(policies)}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes_flat = axes.flatten()

for i, car in enumerate(cars):
    ax     = axes_flat[i]
    car_df = df_plot[df_plot.car_name == car]
    v      = float(car_df['v'].iloc[0])
    M      = float(car_df['M'].iloc[0])

    for p in policies:
        sub = car_df[car_df.policy == p]
        if sub.empty:
            continue
        pvt  = sub.pivot_table(index='round', columns='run_id',
                                values='S', aggfunc='mean').sort_index()
        mean = pvt.mean(axis=1)
        sd   = pvt.std(axis=1)
        ax.plot(pvt.index, mean, color=colors[p], lw=1.8, label=p)
        ax.fill_between(pvt.index, mean - sd, mean + sd,
                        color=colors[p], alpha=0.15, linewidth=0)

    ax.axhline(v, color='gray', ls='--', lw=1, alpha=0.7)
    ax.axhline(M, color='gray', ls='--', lw=1, alpha=0.7)
    ax.text(16.1, v, ' v', va='center', fontsize=9, color='gray')
    ax.text(16.1, M, ' M', va='center', fontsize=9, color='gray')

    veh_label = car_info.loc[car, 'vehicle']
    ax.set_title(f'{veh_label}: {car}\n$v$ = \${v/1000:,.1f}k    $M$ = \${M/1000:,.1f}k    ($v/M$ = {v/M:.2f})',
                 fontsize=10)
    ax.set_xlabel('round')
    ax.set_ylabel('seller ask')
    ax.set_xlim(1, 16)
    pad = (M - v) * 0.15
    ax.set_ylim(v - pad, M + pad)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:,.0f}k'))
    if i == 0:
        ax.legend(fontsize=8, loc='lower left', ncol=2, framealpha=0.9)

# Sixth panel: violin of normalized asks, pooled over cars and rounds
ax    = axes_flat[5]
data  = [df_plot[df_plot.policy == p]['S_norm'].dropna().values for p in policies]
parts = ax.violinplot(data, positions=np.arange(len(policies)),
                      showmeans=True, showmedians=False, widths=0.8)
for j, body in enumerate(parts['bodies']):
    body.set_facecolor(colors[policies[j]])
    body.set_edgecolor('black')
    body.set_alpha(0.6)
for key in ('cbars', 'cmins', 'cmaxes', 'cmeans'):
    if key in parts:
        parts[key].set_edgecolor('black')
        parts[key].set_linewidth(1.0)
ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.7)
ax.axhline(1, color='gray', ls='--', lw=1, alpha=0.7)
ax.text(len(policies) - 0.4, 0, ' v', va='center', fontsize=9, color='gray')
ax.text(len(policies) - 0.4, 1, ' M', va='center', fontsize=9, color='gray')
ax.set_xticks(np.arange(len(policies)))
ax.set_xticklabels(policies, rotation=25, ha='right')
ax.set_ylabel(r'normalized seller ask  $(S - v)/(M - v)$')
ax.set_title('distribution by policy\n(normalized; all cars, all rounds)')
ax.set_ylim(-0.1, 1.15)

plt.tight_layout()
plt.savefig('figE_trajectories.png', dpi=140, bbox_inches='tight')
plt.show()

## 7. Agreement and truncation sanity statistics

In [ ]:
# Per-policy overall rates
summary = (clean.groupby('policy')
               .agg(n_turns            = ('round',            'size'),
                    n_runs             = ('run_id',           'nunique'),
                    pct_lower_trunc    = ('lower_truncated',  lambda s: 100 * s.mean()),
                    pct_upper_trunc    = ('upper_truncated',  lambda s: 100 * s.mean()),
                    pct_agreement_turn = ('agreement_flag',   lambda s: 100 * s.mean()))
               .reindex(POLICY_ORDER)
               .round(2))
summary